# Bronze Layer
Bronze layer is typically your first "layer" or schema where the data lands as is. If you're familiar with database design and concepts, bronze usually equated to the "staging" or stg layer. There are numerous ways this can be written to bronze: you can keep all the format as string and auto-create a table on write, pre-determine the schema and overwrite/append, or rarer cases, pre-transform the data and write to bronze.

whatever the case maybe, you can think of bronze as the landing zone in a Databricks SQL Warehouse from whatever "source" your data is housed.

In this notebook example, we'll assume a source is from an API.

In [0]:
import requests
import json

# parameters for sample osrs ge data
ge_url = "https://prices.runescape.wiki/api/v1/osrs/latest"
response = requests.get(ge_url)
ge_data = response.json()["data"]

# set records to empty list; for each item_id, get the item info for each id and append.
records = []
for item_id, item_info in ge_data.items():
    item_info["id"] = int(item_id)
    records.append(item_info)

# write to spark_df using spark.createDataFrame() function
spark_df = spark.createDataFrame(records)


In [0]:
#using display() function as a better way to check how the data is written
display(spark_df)

In [0]:
# now write the spark_df to delta table in databricks sql warehouse
# you can change "overwrite" to "append" if you feel that you need incrementally add to this table. refer to this documentation for more options: https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html
spark_df.write.format("delta").mode("overwrite").saveAsTable("ga_dev.bronze_metastore_osrs.ge_prices")

In [0]:
%sql
-- magic %sql to do a quick check
-- select count(*) from ga_dev.bronze_metastore_osrs.ge_prices
select * from ga_dev.bronze_metastore_osrs.ge_prices limit 1000;